# NASA Deep Space Image Classification & Neural Training Pipeline

**Author:** Raúl Salas Sahuquillo  
**Repository:** [https://github.com/RaulSalasSahuquillo/nasa-deep-space-classifier](https://github.com/RaulSalasSahuquillo/nasa-deep-space-classifier)  
**License:** Creative Commons Attribution-NonCommercial-ShareAlike 4.0 (CC BY-NC-SA 4.0)  
**Notebook:** `training/network_training.ipynb`  
**Description:** PyTorch Transfer Learning pipeline using ResNet-34 for astronomical target classification with AMP, Matplotlib learning curves, and test evaluation.

In [ ]:
# ==============================================================================
# NASA Deep Space Image Classification & Neural Training Pipeline
# File: training/network_training.ipynb
# Author: Raúl Salas Sahuquillo
# ==============================================================================

import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision.models import resnet34, ResNet34_Weights

from google.colab import files

In [ ]:
# Custom Dataset Definition
class CosmosDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = int(self.df.iloc[idx]['label'])
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

# Augmentations & DataLoaders Setup
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(180),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_data = CosmosDataset('/content/cosmos_train.csv', transform=train_transform)
test_data = CosmosDataset('/content/cosmos_test.csv', transform=test_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

# Model Architecture & Acceleration Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training hardware device: {device}\n")

class_names = ['star', 'galaxy', 'quasar', 'nebula', 'planet']
net = resnet34(weights=ResNet34_Weights.DEFAULT)
num_ftrs = net.fc.in_features
net.fc = nn.Linear(num_ftrs, len(class_names))  # 5 Astronomical Classes
net = net.to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.0001)
scaler = torch.cuda.amp.GradScaler()

# Training Loop Execution with Metrics History Tracking
epochs = 25
history = {
    'train_loss': [],
    'train_acc': []
}

print("STARTING RESNET34 TRAINING LOOP")

for epoch in range(epochs):
    net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = net(inputs)
            loss = loss_function(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)
    
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(acc)
    
    print(f"Epoch [{epoch+1:02d}/{epochs}] | Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")

# Save weights artifact
weights_path = '/content/trained_net.pth'
torch.save(net.state_dict(), weights_path)
print(f"\nTraining finished! Weights file saved to '{weights_path}'")

In [ ]:
# Matplotlib Visualizations: Loss & Accuracy Learning Curves

epoch_range = range(1, epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Loss Curve
ax1.plot(epoch_range, history['train_loss'], 'o-', color='#1f77b4', linewidth=2, label='Training Loss')
ax1.set_title('ResNet-34 Training Loss per Epoch', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('CrossEntropy Loss', fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper right')

# Accuracy Curve
ax2.plot(epoch_range, history['train_acc'], 's-', color='#2ca02c', linewidth=2, label='Training Accuracy')
ax2.set_title('ResNet-34 Classification Accuracy (%)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Accuracy (%)', fontsize=11)
ax2.set_ylim([0, 105])
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='lower right')

plt.suptitle('NASA Deep Space Classifier — ResNet-34 Learning Progress', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

# Save figure
plt.savefig('/content/learning_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate Performance on Test Split & Matplotlib Confusion Matrix

net.eval()
all_preds = []
all_labels = []
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = 100 * correct / total
print(f"Final Test Model Accuracy: {test_acc:.2f}%\n")

# Confusion Matrix Calculation
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds)

# Matplotlib Confusion Matrix Heatmap
fig, ax = plt.subplots(figsize=(7, 6), dpi=150)
cax = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('ResNet-34 Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
plt.colorbar(cax)

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, fontsize=10, fontweight='bold')
plt.yticks(tick_marks, class_names, fontsize=10, fontweight='bold')

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black",
                 fontweight="bold")

plt.ylabel('True Class', fontsize=11)
plt.xlabel('Predicted Class', fontsize=11)
plt.tight_layout()

plt.savefig('/content/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Trigger browser download of trained weights and figures
print("Downloading 'trained_net.pth', 'learning_curves.png', and 'confusion_matrix.png'...")
files.download('/content/trained_net.pth')
files.download('/content/learning_curves.png')
files.download('/content/confusion_matrix.png')